In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'huggingface-hub>=0.26.0', 'python-dotenv>=1.0.0',
    'pyyaml>=6.0', 'requests>=2.32.0',
    'torch>=2.3.0', 'torchaudio>=2.3.0',
    'soundfile>=0.12.1', 'numpy>=1.26.0',
    'sentencepiece>=0.2.0', 'moshi>=0.1.0',
    'transformers>=4.40.0',
], check=True)
subprocess.run(['apt-get', 'install', '-qq', '-y', 'ffmpeg'], check=True)

In [ ]:
import os, json, time, threading, random
from pathlib import Path
from datetime import datetime
import yaml, requests
import numpy as np
import soundfile as sf
import torch
import torchaudio
from huggingface_hub import HfApi

WORK_DIR        = Path('/kaggle/working')
SYNTH_DIR       = WORK_DIR / 'synthesized_audio'
CHECKPOINT_PATH = WORK_DIR / 'checkpoint_p5a.json'
CONFIG_DIR      = Path('/kaggle/input/datasets/mirza176528/s2s-pipline-v2-0-2/config')

SYNTH_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SR  = 24000
TARGET_LUFS = -23.0
SAVE_EVERY  = 50
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'[config] device={DEVICE}')

In [ ]:
def load_secrets():
    try:
        from kaggle_secrets import UserSecretsClient
        c = UserSecretsClient()
        s = {k: c.get_secret(k) for k in ['HF_TOKEN_PRIMARY','HF_TOKEN_SECONDARY','HF_TOKEN_TERTIARY']}
        print('[secrets] Kaggle'); return s
    except Exception: pass
    env_file = Path('.env')
    if env_file.exists():
        from dotenv import load_dotenv; load_dotenv(env_file)
    required = ['HF_TOKEN_PRIMARY','HF_TOKEN_SECONDARY','HF_TOKEN_TERTIARY']
    missing = [k for k in required if not os.environ.get(k)]
    if missing: raise RuntimeError(f'Missing: {missing}')
    return {k: os.environ[k] for k in required}

SECRETS  = load_secrets()
HF_TOKEN = SECRETS['HF_TOKEN_PRIMARY']

with open(CONFIG_DIR / 'hf_repos.yaml') as f: repos_cfg = yaml.safe_load(f)
STAGE0_REPO  = repos_cfg['repos']['stage0_codec']['repo_id']
STAGE3_REPO  = repos_cfg['repos']['stage3_agent']['repo_id']
STAGE45_REPO = repos_cfg['repos']['stage45_e2e']['repo_id']
HF_API       = HfApi(token=HF_TOKEN)
print(f'[config] stage0:  {STAGE0_REPO}')
print(f'[config] stage3:  {STAGE3_REPO}')
print(f'[config] stage45: {STAGE45_REPO}')

In [ ]:
MODEL_CKPT_DIR = WORK_DIR / 'model_checkpoint'
MODEL_CKPT_DIR.mkdir(parents=True, exist_ok=True)

STAGE4_CKPT_REPO = repos_cfg['repos']['stage45_e2e']['repo_id']

ckpt_files = ['rq_transformer.pt', 'mimi_finetuned.pt', 'tokenizer.model']
for fname in ckpt_files:
    dest = MODEL_CKPT_DIR / fname
    if dest.exists():
        print(f'[ckpt] {fname} already downloaded')
        continue
    print(f'[ckpt] downloading {fname}...')
    for attempt in range(8):
        try:
            url = f'https://huggingface.co/datasets/{STAGE4_CKPT_REPO}/resolve/main/checkpoints/{fname}'
            r = requests.get(url, headers={'Authorization': f'Bearer {HF_TOKEN}'}, timeout=300, stream=True)
            r.raise_for_status()
            with open(dest, 'wb') as f:
                for chunk in r.iter_content(chunk_size=1024*1024): f.write(chunk)
            print(f'[ckpt] {fname} downloaded ({dest.stat().st_size/1024**2:.1f} MB)')
            break
        except Exception as e:
            wait = min(2**attempt, 120)
            print(f'  attempt {attempt+1}/8 failed: {e} — retry in {wait}s')
            time.sleep(wait)

print('\n[model] loading RQ-Transformer checkpoint...')
from moshi.models import loaders

mimi_model = loaders.get_mimi(str(MODEL_CKPT_DIR), device=DEVICE)
mimi_model.eval()
for p in mimi_model.parameters(): p.requires_grad = False

rq_state = torch.load(str(MODEL_CKPT_DIR / 'rq_transformer.pt'), map_location=DEVICE)
from moshi.models.lm import LMModel
model_config = rq_state.get('model_config') or rq_state.get('config') or rq_state.get('model_args')
if model_config is None:
    raise RuntimeError(f"Checkpoint has no model config. Available keys: {list(rq_state.keys())}")
rq_model = LMModel(**model_config)
rq_model.load_state_dict(rq_state['model'])
rq_model = rq_model.to(DEVICE).eval()
for p in rq_model.parameters(): p.requires_grad = False

import sentencepiece as spm
tokenizer = spm.SentencePieceProcessor()
tokenizer.Load(str(MODEL_CKPT_DIR / 'tokenizer.model'))

print(f'[model] all components loaded on {DEVICE}')

In [ ]:
AGENT_VOICE_SEED  = 42
USER_VOICE_SEEDS  = [1, 7, 13, 21, 37, 55, 71, 89]

def text_to_mimi_tokens(text, voice_seed=42, temperature=0.8, top_p=0.95):
    token_ids = tokenizer.Encode(text)
    text_tensor = torch.LongTensor(token_ids).unsqueeze(0).to(DEVICE)

    torch.manual_seed(voice_seed)
    with torch.no_grad():
        generated = rq_model.generate(
            text_tokens=text_tensor,
            max_new_tokens=512,
            temperature=temperature,
            top_p=top_p,
        )
    return generated


def mimi_tokens_to_wav(tokens):
    with torch.no_grad():
        audio = mimi_model.decode(tokens)
    audio_np = audio.squeeze().cpu().numpy()
    if audio_np.ndim > 1:
        audio_np = audio_np.mean(axis=0)
    return audio_np


def normalize_loudness(audio, sr=TARGET_SR):
    import pyloudnorm as pyln
    meter = pyln.Meter(sr)
    loudness = meter.integrated_loudness(audio.astype(np.float64))
    if not np.isfinite(loudness):
        return audio
    normalized = pyln.normalize.loudness(audio.astype(np.float64), loudness, TARGET_LUFS)
    if np.max(np.abs(normalized)) > 1.0:
        normalized = normalized / np.max(np.abs(normalized)) * 0.95
    return normalized.astype(np.float32)


def synthesize_turn(text, voice_seed, episode_id, turn_id):
    out_path = SYNTH_DIR / f'{episode_id}_t{turn_id}.wav'
    if out_path.exists() and out_path.stat().st_size > 1000:
        return out_path
    tokens  = text_to_mimi_tokens(text, voice_seed=voice_seed)
    audio   = mimi_tokens_to_wav(tokens)
    audio   = normalize_loudness(audio)
    sf.write(str(out_path), audio, TARGET_SR, subtype='PCM_16')
    return out_path


def encode_wav_to_tokens(wav_path):
    audio, sr = sf.read(str(wav_path), dtype='float32')
    if sr != TARGET_SR:
        raise ValueError(f'Expected {TARGET_SR}Hz got {sr}Hz')
    tensor = torch.FloatTensor(audio).unsqueeze(0).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        codes = mimi_model.encode(tensor)
    return codes.squeeze(0).cpu().numpy().T


print('[tts] synthesis functions ready')

In [ ]:
def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        try:
            with open(CHECKPOINT_PATH) as f: state = json.load(f)
            print(f'[checkpoint] local — synthesized={state["stats"]["synthesized"]}')
            return state
        except Exception: pass
    try:
        url = f'https://huggingface.co/datasets/{STAGE45_REPO}/resolve/main/checkpoint_p5a.json'
        r = requests.get(url, headers={'Authorization': f'Bearer {HF_TOKEN}'}, timeout=30)
        if r.status_code == 200:
            state = r.json()
            with open(CHECKPOINT_PATH, 'w') as f: json.dump(state, f)
            print(f'[checkpoint] HF fallback — synthesized={state["stats"]["synthesized"]}')
            return state
    except Exception: pass
    print('[checkpoint] fresh start')
    return {
        'done_ids': [],
        'stats': {'synthesized': 0, 'failed': 0, 'turns_processed': 0},
        'last_updated': None,
    }

cp_lock = threading.Lock()

def save_checkpoint(state, upload=False):
    with cp_lock:
        state['last_updated'] = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
        tmp = str(CHECKPOINT_PATH) + '.tmp'
        with open(tmp, 'w') as f: json.dump(state, f)
        os.replace(tmp, str(CHECKPOINT_PATH))
    if not upload: return
    for attempt in range(6):
        try:
            HF_API.upload_file(path_or_fileobj=json.dumps(state).encode(),
                path_in_repo='checkpoint_p5a.json', repo_id=STAGE45_REPO,
                repo_type='dataset', commit_message='p5a checkpoint')
            return
        except Exception: time.sleep(min(2**attempt, 60))

state    = load_checkpoint()
done_set = set(state['done_ids'])

In [ ]:
episodes_local = WORK_DIR / 'agent_episodes.jsonl'
if not episodes_local.exists():
    print('[episodes] downloading from HF stage3...')
    for attempt in range(6):
        try:
            url = f'https://huggingface.co/datasets/{STAGE3_REPO}/resolve/main/agent_episodes.jsonl'
            r = requests.get(url, headers={'Authorization': f'Bearer {HF_TOKEN}'}, timeout=120, stream=True)
            r.raise_for_status()
            with open(episodes_local, 'wb') as f:
                for chunk in r.iter_content(chunk_size=65536): f.write(chunk)
            print('[episodes] downloaded')
            break
        except Exception as e:
            time.sleep(min(2**attempt, 60))

all_episodes = []
with open(episodes_local, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line: all_episodes.append(json.loads(line))

pending = [ep for ep in all_episodes if ep['episode_id'] not in done_set]
print(f'[p5a] total={len(all_episodes)} done={len(done_set)} pending={len(pending)}')

updated_episodes_path = WORK_DIR / 'agent_episodes_with_audio.jsonl'

for idx, episode in enumerate(pending):
    ep_id       = episode['episode_id']
    agent_seed  = AGENT_VOICE_SEED
    user_seed   = random.choice(USER_VOICE_SEEDS)

    try:
        for turn in episode['turns']:
            tid  = turn['turn_id']
            text = turn.get('transcript_urdu', '')

            if not text or not text.strip():
                continue

            if turn.get('action_type') == 'tool_call':
                turn['audio_token_path'] = None
                continue

            seed = agent_seed if turn['speaker'] == 'agent' else user_seed
            wav_path = synthesize_turn(text, seed, ep_id, tid)
            tokens   = encode_wav_to_tokens(wav_path)

            token_path = SYNTH_DIR / f'{ep_id}_t{tid}_tokens.npy'
            np.save(str(token_path), tokens)
            turn['audio_token_path'] = f'audio/{ep_id}_t{tid}_tokens.npy'

            with cp_lock:
                state['stats']['turns_processed'] += 1

            wav_path.unlink(missing_ok=True)

        with open(updated_episodes_path, 'a', encoding='utf-8') as f:
            f.write(json.dumps(episode, ensure_ascii=False) + '\n')

        with cp_lock:
            state['stats']['synthesized'] += 1
            done_set.add(ep_id)
            state['done_ids'].append(ep_id)

    except Exception as e:
        print(f'  [error] {ep_id}: {e}')
        with cp_lock:
            state['stats']['failed'] += 1
            done_set.add(ep_id)
            state['done_ids'].append(ep_id)

    if (idx + 1) % SAVE_EVERY == 0 or idx + 1 == len(pending):
        upload_now = (idx + 1) % (SAVE_EVERY * 5) == 0
        save_checkpoint(state, upload=upload_now)
        print(f'  [{idx+1}/{len(pending)}] synthesized={state["stats"]["synthesized"]} '
              f'turns={state["stats"]["turns_processed"]} failed={state["stats"]["failed"]}')

save_checkpoint(state, upload=True)
total_done = sum(1 for _ in open(updated_episodes_path, encoding='utf-8')) if updated_episodes_path.exists() else 0
print(f'\n[p5a] synthesis complete — {total_done} episodes with audio paths')
print('[done] ready for p5b_interleave.ipynb')